# DEMreg 사용 예시

이 노트북은 `demreg` 모듈(`dn2dem_pos`)을 실제로 호출하는 최소 예시입니다.

**시나리오**
1. SDO/AIA 6채널 온도 응답함수를 로드
2. 알려진 가우시안 모양의 DEM을 가정 → 합성 DN 생성 (forward)
3. `dn2dem_pos`로 DN에서 DEM을 복원 (inverse)
4. 입력 DEM vs 복원 DEM 비교
5. 작은 2D 맵 예시

노트북은 demreg 디렉토리(이 파일이 있는 위치)에서 실행한다고 가정합니다.

> **참고**: 0D 단일 픽셀 경우 안정적인 수렴을 위해 `dem_norm0`을 명시적으로 (예: 1로 채운 배열) 넣어 주는 것을 권장. 안 주면 self-norm 2-pass 경로가 일부 입력에서 SVD 발산할 수 있음.

## 0. 의존성

In [ ]:
# 필요 시 설치
# !pip install numpy matplotlib tqdm threadpoolctl

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

# demreg 패키지를 import 가능하게 하기 위해 부모 경로(core/)를 sys.path에 추가
HERE = os.path.abspath('')
CORE_DIR = os.path.dirname(HERE)            # .../08-DEM-calc/core
if CORE_DIR not in sys.path:
    sys.path.insert(0, CORE_DIR)

from demreg import dn2dem_pos
print('demreg loaded from:', dn2dem_pos.__module__)

## 1. 온도 응답함수 로드 (AIA 6채널)

In [ ]:
RESP_DIR = os.path.join(CORE_DIR, 'response')
tresp = np.load(os.path.join(RESP_DIR, 'tresp_aia.npy'))        # (81, 6)
tresp_logt = np.load(os.path.join(RESP_DIR, 'tresp_logt.npy'))  # (81,)
channels = ['94', '131', '171', '193', '211', '335']

print('tresp:', tresp.shape, '  tresp_logt:', tresp_logt.shape,
      '  logT range:', tresp_logt.min(), '–', tresp_logt.max())

fig, ax = plt.subplots(figsize=(7, 4))
for i, ch in enumerate(channels):
    ax.semilogy(tresp_logt, tresp[:, i], label=f'AIA {ch} Å')
ax.set_xlabel('log T [K]'); ax.set_ylabel('Response')
ax.set_xlim(5.5, 7.5); ax.set_ylim(1e-29, 1e-23)
ax.legend(ncol=3, fontsize=8); ax.set_title('AIA Temperature Response')
plt.tight_layout(); plt.show()

## 2. 합성 DEM 정의 & forward로 DN 생성

두 개의 가우시안 성분 (logT 6.1 — 활동영역 평온 / logT 6.5 — 핫 컴포넌트)을 가지는
테스트용 DEM을 만들고, 응답함수로 곱해 채널별 DN을 합성합니다.

In [ ]:
# DEM 온도 그리드 (선형 K). bin edges 기준.
# 응답함수가 잘 정의된 logT 5.7–7.3 범위로 잡는다 (총 41 bin).
temps = np.logspace(5.7, 7.3, 42)
logt_mid = 0.5 * (np.log10(temps[1:]) + np.log10(temps[:-1]))
dlogt = np.diff(np.log10(temps))
nt = len(dlogt)

def gaussian(x, mu, sig, amp):
    return amp * np.exp(-0.5 * ((x - mu) / sig) ** 2)

dem_true = (gaussian(logt_mid, 6.10, 0.10, 2.0e22) +
            gaussian(logt_mid, 6.50, 0.08, 5.0e21))

# 합성 DN: g(f) = sum_T K(f,T) * DEM(T) * dT  ;  dT = T * ln(10) * dlogT
# tresp를 logt_mid에 맞춰 로그-공간에서 보간
tr = np.empty((nt, 6))
for i in range(6):
    tr[:, i] = 10 ** np.interp(logt_mid, tresp_logt, np.log10(tresp[:, i]))

dT = (10 ** logt_mid) * np.log(10) * dlogt
dn_true = (tr * dT[:, None] * dem_true[:, None]).sum(axis=0)   # (6,)

# 5% 가우시안 노이즈 + Poisson floor
rng = np.random.default_rng(1)
edn = np.sqrt(np.maximum(dn_true, 1)) + 0.05 * dn_true
dn  = dn_true + edn * rng.standard_normal(6)

for i, ch in enumerate(channels):
    print(f'  {ch:>4} Å:  DN = {dn[i]:10.3f}  ±{edn[i]:8.3f}   (true {dn_true[i]:10.3f})')

## 3. `dn2dem_pos`로 DEM 역산 (0D — 단일 픽셀)

`dem_norm0`을 ones 배열로 명시적으로 넣어 안정적인 수렴을 보장합니다.

In [ ]:
dem_norm0 = np.ones(nt)            # 초기 L 가중치 = flat

dem, edem, elogt, chisq, dn_reg = dn2dem_pos(
    dn, edn, tresp, tresp_logt, temps,
    reg_tweak=1.0,    # 목표 reduced chi^2
    max_iter=15,      # positivity loop 한도
    rgt_fact=1.5,     # 매 반복마다 chi^2 목표 *1.5
    dem_norm0=dem_norm0,
)

print(f'chisq = {chisq:.3f}')
print(f'dn_reg (재구성) = {dn_reg}')
print(f'dn       (입력) = {dn}')

## 4. 결과 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(logt_mid, dem_true, 'k-', lw=2, label='True DEM')
ax.errorbar(logt_mid, dem, yerr=edem, xerr=elogt,
            fmt='o', ms=4, color='C3', capsize=2, label='Recovered')
ax.set_xlabel('log T [K]')
ax.set_ylabel(r'DEM  [cm$^{-5}$ K$^{-1}$]')
ax.set_yscale('log')
ax.set_ylim(1e18, 1e23)
ax.set_title(f'DEM 복원 (chi^2 = {chisq:.2f})')
ax.legend()

ax = axes[1]
x = np.arange(6)
ax.errorbar(x, dn, yerr=edn, fmt='ks', ms=8, label='Input DN', capsize=3)
ax.plot(x, dn_reg, 'rD', ms=8, label='Reconstructed (K·DEM)')
ax.set_xticks(x); ax.set_xticklabels([f'{c}Å' for c in channels])
ax.set_ylabel('DN / px / s'); ax.set_yscale('log')
ax.set_title('채널별 입력 vs 재구성')
ax.legend()

plt.tight_layout(); plt.show()

## 5. 작은 2D 맵 예시 (4×4 픽셀)

픽셀별로 DEM 진폭을 다르게 한 4×4 가짜 맵을 만든 뒤, `dn2dem_pos`에 `(nx, ny, nf)` 모양으로 통째로 넣어 한 번에 처리합니다.

In [ ]:
nx, ny = 4, 4
amp_map = np.linspace(0.5, 2.0, nx * ny).reshape(nx, ny)

dem_map_true = dem_true[None, None, :] * amp_map[:, :, None]     # (nx, ny, nt)
dn_map_true  = np.einsum('xyt,tf->xyf', dem_map_true * dT[None, None, :], tr)

edn_map = np.sqrt(np.maximum(dn_map_true, 1)) + 0.05 * dn_map_true
dn_map  = dn_map_true + edn_map * rng.standard_normal(dn_map_true.shape)

# 맵 전체에 대한 초기 가중치 (모든 픽셀, 모든 T bin = 1)
dem_norm0_map = np.ones((nx, ny, nt))

dem_m, edem_m, elogt_m, chisq_m, dn_reg_m = dn2dem_pos(
    dn_map, edn_map, tresp, tresp_logt, temps,
    reg_tweak=1.0, max_iter=15, rgt_fact=1.5,
    dem_norm0=dem_norm0_map,
)

print('dem_map shape:', dem_m.shape, '  chisq map:')
print(chisq_m)

In [ ]:
# 특정 logT bin 한 장을 맵으로 시각화: 'true 진폭 맵'과 비교
k_peak = int(np.argmin(np.abs(logt_mid - 6.10)))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
im0 = axes[0].imshow(dem_map_true[:, :, k_peak], origin='lower')
axes[0].set_title(f'True DEM @ logT={logt_mid[k_peak]:.2f}'); plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(dem_m[:, :, k_peak], origin='lower')
axes[1].set_title('Recovered'); plt.colorbar(im1, ax=axes[1])
im2 = axes[2].imshow(chisq_m, origin='lower', cmap='magma')
axes[2].set_title(r'$\chi^2$ map'); plt.colorbar(im2, ax=axes[2])
plt.tight_layout(); plt.show()

## 6. 옵션 빠른 참고

| 옵션 | 언제 만지나 |
|---|---|
| `reg_tweak` | DEM이 너무 "흐리면" 낮춰서(엄격) / 음수해가 자주 나오면 키워서(완화) |
| `max_iter` / `rgt_fact` | 양수해 못 찾을 때 늘려본다 (`max_iter`↑ 또는 `rgt_fact`↑) |
| `dem_norm0` | 미리 알고 있는 DEM 모양이 있으면 강력하게 권장. 단일 픽셀(0D)에서는 가급적 명시 |
| `gloci=1` | `dem_norm0` 없을 때 EM-loci 기반 가중치 사용 |
| `emd_int=True, l_emd=True` | 고온부(>10⁶·⁵ K) 잘 안 풀릴 때 시도 |
| `non_pos=True` | 양수 강제 없이 첫 해만 보고 싶을 때 |

자세한 설명은 같은 디렉토리의 `USAGE.md` 참조.